# Week 6: Constrained Gradient Unlearning

This notebook runs the Week 6 PCGrad-style constrained-gradient unlearning sweep. It clones the GitHub repository, runs the Week 6 runner, writes outputs under `Week 6/results/constrained_gradient_unlearning_v1`, and pushes those outputs back to GitHub.

## 1. Install Runtime Dependencies

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes pandas numpy safetensors

## 2. Clone Or Update The GitHub Repo

Add `GITHUB_TOKEN` in Colab Secrets before running this cell. The fallback branch lets this notebook run from the draft PR before it is merged; after merge, `main` is used normally.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = "HannanSeyfi/unlearning-thesis"
BRANCH = "main"
FALLBACK_BRANCH = "codex/week6-constrained-gradient-unlearning"
REPO_DIR = Path("/content/unlearning-thesis")

if not (REPO_DIR / ".git").exists():
    !git clone --branch {BRANCH} https://github.com/{REPOSITORY}.git {REPO_DIR}

%cd {REPO_DIR}
from Tools.github_colab_sync import setup_colab_repo, commit_and_push

THESIS_DIR = setup_colab_repo(REPOSITORY, BRANCH, REPO_DIR)
%cd {THESIS_DIR}

SCRIPT_PATH = THESIS_DIR / "Week 6" / "constrained_gradient_unlearning" / "train_week6_constrained_gradient_unlearning.py"
if not SCRIPT_PATH.exists():
    print(f"Week 6 script missing on {BRANCH}; checking out fallback branch {FALLBACK_BRANCH}.")
    subprocess.run(["git", "fetch", "origin", FALLBACK_BRANCH], cwd=THESIS_DIR, check=True)
    subprocess.run(["git", "checkout", FALLBACK_BRANCH], cwd=THESIS_DIR, check=True)
    BRANCH = FALLBACK_BRANCH
    SCRIPT_PATH = THESIS_DIR / "Week 6" / "constrained_gradient_unlearning" / "train_week6_constrained_gradient_unlearning.py"

print("Using branch:", BRANCH)
print("Week 6 script:", SCRIPT_PATH)

## 3. Run The Focused Week 6 Sweep

The default focused sweep is designed for a first Colab pass. Set `RUN_FULL_GRID = True` only after the focused run confirms the constrained update behaves sensibly.

In [ ]:
RUN_FULL_GRID = False
RESET_EXISTING_RUN = False
MAX_EPOCHS = 6
PUSH_EACH_EPOCH = True

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--repo-root",
    str(THESIS_DIR),
    "--max-epochs",
    str(MAX_EPOCHS),
    "--push-branch",
    BRANCH,
]
if RUN_FULL_GRID:
    cmd.append("--run-full-grid")
if RESET_EXISTING_RUN:
    cmd.append("--reset")
if PUSH_EACH_EPOCH:
    cmd.append("--push-each-epoch")

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=THESIS_DIR, check=True)

## 4. Inspect The Results

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

RUN_DIR = THESIS_DIR / "Week 6" / "results" / "constrained_gradient_unlearning_v1"
RESULTS_DIR = RUN_DIR / "results"

comparison = pd.read_csv(RESULTS_DIR / "week4_week5_week6_comparison.csv")
display(comparison)
display(Markdown((RESULTS_DIR / "week6_constrained_gradient_report.md").read_text(encoding="utf-8")))

## 5. Final GitHub Sync

The runner pushes after each epoch when `PUSH_EACH_EPOCH = True`. This final sync is harmless if there are no remaining changes.

In [ ]:
commit_and_push(
    RUN_DIR,
    "Colab: save Week 6 constrained gradient outputs",
    repo_dir=THESIS_DIR,
    branch=BRANCH,
)